# L12 — A/B testing & multi-armed bandits for rankers
**Objective 15.**

**Northfield Grocers context:** Northfield's recommendation team has a new substitution ranker (B) for out-of-stock items. A fixed A/B test sends half of shoppers to the worse ranker for two weeks; a Thompson-sampling bandit shifts traffic as evidence arrives. Run both on the click log and write the governance record that accompanies the champion-alias change.

**Retail use cases:** Rolling out a substitution ranker with bounded regret; comparing two promo-copy models on click-through.

**Platform:** Databricks (no GPU). Fully executable in SMOKE mode.

**Done means:** bandit converges to the planted better arm; cumulative regret below A/B; decision record written.

## Step 1 — The click log
*Why:* arm A and B are two ranker versions. The log has a planted truth (D6): B's CTR is ~30 % higher. Neither method may peek — they only see clicks as they arrive.

In [ ]:
# === Dependency check (no installs, no restarts — see L00_setup for one-time cluster setup) ===
import importlib.util
_REQ = ["numpy", "pandas", "sklearn", "onnx", "onnxruntime", "skl2onnx"]
_missing = [m for m in _REQ if importlib.util.find_spec(m) is None]
if _missing:
    raise ImportError(f"Missing packages {_missing}. Run labs/L00_Setup/L00_setup.ipynb once on this cluster/venv "
                      f"(or `%pip install {' '.join(_missing)}` in a new cell, then restart Python and Run All from the top).")
print("All lab dependencies present.")

In [ ]:
# === Lab environment header (identical in every lab) ===
import os, sys, json, time, math, shutil, re
import numpy as np, pandas as pd
# Mode: "GPU" runs the full lab on Azure GPU compute; "SMOKE" runs the CPU/synthetic path anywhere.
LAB_MODE = os.environ.get("LAB_MODE") or ("GPU" if shutil.which("nvidia-smi") else "SMOKE")
# Data folder: env override → package-relative (../../data) → Databricks Unity Catalog volume
_candidates = [os.environ.get("DATA_DIR"), os.path.abspath(os.path.join(os.getcwd(), "..", "..", "data")), "/Volumes/northfield/llmops/labdata"]
DATA_DIR = next((c for c in _candidates if c and os.path.exists(os.path.join(c, "catalog_items.csv"))), None)
if DATA_DIR is None:
    raise FileNotFoundError("Lab data not found. Set os.environ['DATA_DIR'] to the folder containing catalog_items.csv "
                            "(e.g. a Unity Catalog volume path) in a cell above this one.")
def gpu_only(msg):
    """Called wherever a step needs a GPU / model download that the smoke path cannot run."""
    print(f"[{LAB_MODE}] GPU-only step not executed here: {msg}")
def check(cond, msg):
    """Binary 'done means' assertion — prints PASS/FAIL and raises on FAIL so the notebook stops."""
    print(("PASS " if cond else "FAIL ") + msg); assert cond, msg
print(f"LAB_MODE={LAB_MODE}  DATA_DIR={DATA_DIR}")

In [ ]:
log = pd.read_csv(os.path.join(DATA_DIR, "click_log.csv"))
truth = log.groupby("arm").click.mean(); print("true CTRs (hidden from the algorithms):", truth.round(4).to_dict())

## Step 2 — Fixed-split A/B test with a significance check
*Why:* the classic approach: 50/50 for N impressions, then a two-proportion z-test. Regret = impressions sent to the worse arm × CTR gap.

In [ ]:
from scipy.stats import norm
def ab_test(log, n):
    # TODO: take first n rows per arm alternating order; compute CTRs, pooled z-test, p-value, regret
    raise NotImplementedError('complete this step')

ab = ab_test(log, 5000); print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in ab.items()})
check(ab["winner"] == "B", "A/B identifies B")

## Step 3 — Thompson sampling bandit
*Why:* keep a Beta(α, β) posterior per arm; each impression, sample from both, serve the larger. Traffic shifts to the better arm as evidence accumulates, so regret grows sub-linearly.

In [ ]:
def thompson(log, n_impressions, seed=0):
    """Replay: at each step choose an arm, consume the next unseen click for that arm. Returns (allocation, regret)."""
    # TODO: alpha/beta dicts; pools per arm; loop: sample, pick argmax, pop click, update; regret += gap if worse arm
    raise NotImplementedError('complete this step')

served, regret_ts = thompson(log, 10000)
print("bandit allocation:", served, f"| regret {regret_ts:.1f} vs A/B {ab['regret']:.1f}")
check(served["B"] > served["A"] * 2 and regret_ts < ab["regret"], "bandit converges to B with lower regret than A/B")

## Step 4 — Governance record
*Why:* whichever method you use, the decision needs a record: hypothesis, method, sample size, result, decision, owner, date. This is what the model registry alias change (`champion`) links to.

In [ ]:
record = dict(hypothesis="Substitution ranker B raises click-through on out-of-stock substitutions by ≥10% over A", method="A/B z-test + Thompson replay", n_ab=5000, n_bandit=10000,
              ab=ab, bandit=dict(allocation=served, regret=regret_ts), decision="promote B to champion", owner="northfield-recommendation-team", date=time.strftime("%Y-%m-%d"))
os.makedirs("/tmp/l12", exist_ok=True); json.dump(record, open("/tmp/l12/decision.json", "w"), indent=2, default=float)
check(os.path.exists("/tmp/l12/decision.json"), "decision record written"); print("L12 complete.")